<a href="https://colab.research.google.com/github/sde2424242424-coder/2026_spring_assignments/blob/main/quotes_toscrape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on #3 (quotes.toscrape)

본 프로젝트에서는 quotes.toscrape.com 웹사이트에서 1페이지부터 10페이지까지 모든 명언 데이터를 수집하였다. 각 페이지에서 `<span class="text">` 태그를 이용하여 텍스트를 추출하고, 불필요한 특수문자를 제거하여 데이터를 정제하였다. 이후 CountVectorizer를 사용하여 문장을 단어 단위로 분해하고, 각 단어의 등장 여부를 기반으로 수치 데이터로 변환하였다. 변환된 데이터는 행은 명언, 열은 단어, 값은 단어의 등장 횟수를 의미하는 구조를 가진다. 그러나 이 방법은 단어의 순서나 문맥을 고려하지 못하고, 데이터가 희소 행렬 형태로 변환되는 단점이 있다. 이러한 과정을 통해 웹에서 수집한 텍스트 데이터를 분석 가능한 형태로 변환할 수 있음을 확인하였다.

In [8]:
import requests
import pandas as pd
import time
import re

from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer


BASE_URL = "http://quotes.toscrape.com/page/{}/"

In [2]:
def get_quotes():
    all_quotes = []

    for page in range(1, 11):
        url = BASE_URL.format(page)
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        quotes = soup.find_all("span", class_="text")

        for q in quotes:
            text = q.get_text(strip=True)

            # очистка кавычек и лишних символов
            text = re.sub(r"[“”]", "", text)

            all_quotes.append(text)

        print(f"Page {page} collected")
        time.sleep(0.2)

    return all_quotes

In [3]:
def build_dataframe(quotes):
    df = pd.DataFrame({"Quote": quotes})
    return df

In [4]:
def vectorize_text(df):
    vectorizer = CountVectorizer(
        stop_words="english",
        max_features=1000,
        min_df=2
    )

    X = vectorizer.fit_transform(df["Quote"])

    df_vector = pd.DataFrame(
        X.toarray(),
        columns=vectorizer.get_feature_names_out()
    )

    return df_vector

In [9]:
quotes = get_quotes()
df_quotes = build_dataframe(quotes)

print("\nTop 5 quotes:")
print(df_quotes.head())

df_encoded = vectorize_text(df_quotes)

print("\nEncoded DataFrame:")
print(df_encoded.head())

print("\nShape:", df_encoded.shape)

Page 1 collected
Page 2 collected
Page 3 collected
Page 4 collected
Page 5 collected
Page 6 collected
Page 7 collected
Page 8 collected
Page 9 collected
Page 10 collected

Top 5 quotes:
                                               Quote
0  The world as we have created it is a process o...
1  It is our choices, Harry, that show what we tr...
2  There are only two ways to live your life. One...
3  The person, be it gentleman or lady, who has n...
4  Imperfection is beauty, madness is genius and ...

Encoded DataFrame:
   adventure  beautiful  beauty  begin  believe  best  better  big  book  \
0          0          0       0      0        0     0       0    0     0   
1          0          0       0      0        0     0       0    0     0   
2          0          0       0      0        0     0       0    0     0   
3          0          0       0      0        0     0       0    0     0   
4          0          0       1      0        0     0       1    0     0   

   books  ...  try 